In [ ]:
#I heavily recommend running this code in a Jupyter Notebook or similar environment to make package management easier
#This script needs over 50 packages to run, so it is not recommended to run it in a standard Python environment without proper package management.
#
#This script loads customer provided Excel data and shapefile data from Digiroad.
#These files are merged based on the road number and segment number.
#Condition is matched based on the AET and LET values from the shapefile and the Aet and Let values from the Excel file.
#After merging, the script creates a new column "kunto_luokka" based on the "RMS_yhd" values, which represent the road condition.
#Finally, the merged dataset is saved to a GeoPackage file named "road_condition.gpkg", which can be used in QGIS for visualization.
import geopandas as gpd
import pandas as pd
from pathlib import Path

#------Create an output folder if it doesn't exist to save the results.------

#This gets the folder where the script is located, which is assumed to be the "codes" folder in the project structure.
#Made this to work with both Jupyter Notebook and standard Python environments, as __file__ is not defined in Jupyter Notebooks.
try:
    codes_dir = Path(__file__).resolve().parent
except NameError:
    codes_dir = Path.cwd()

#project root directory (one level up from the codes directory)
project_root = codes_dir.parent

#------Create an output folder if it doesn't exist to save the results.------
output_folder = project_root / "output"
output_folder.mkdir(exist_ok=True)

print(output_folder)

#----------Load excel and shapefile data (shapefile is from Digiroad)-----------
#Testing excel file path.
#r"C:\Users\Laptop\Downloads\Paallystettyjen_teiden_lahtotiedot_ominaisuus_kuntotiedot_100m_L145695.xlsx",
excel = pd.read_excel(
    r""
    header=1
)

#----------Shapefile we need is specifically DR_LINKKI.shp, which contains road segments with their attributes.-----------
roads = gpd.read_file(
    r"C:\Users\Laptop\Downloads\KokoSuomi_DIGIROAD_R\PIRKANMAA\PIRKANMAA_2\DR_LINKKI.shp"
)

print("road ORIGINAL", roads.columns.tolist())
print("excel ORIGINAL", excel.columns.tolist())

#-------Filtering by region.-------
#Here we filter the excel data to only include rows where the "Maakunta" column is "Pirkanmaa".
#This was for testing reasons to reduce the dataset size.
#This needs to be modified if we want to have a filter system for the user to select the region of interest.
excel = excel[excel["Maakunta"] == "Pirkanmaa"]

#-------Next, we select only the relevant columns from both datasets and rename them to have consistent naming for merging.-----------
excel = excel[[
    "Tie",
    "Aosa",
    "Aet",
    "Let",
    "RMS_yhd",
    "RIDE_lk", #<-- Is this needed???
    "Ajorata", #<-- Used to separate lanes
    "Kaista" #<-- Used to separate lanes
]].copy()

#-------Aet and Let needs to be renamed to not conflict with the AET and LET columns in the roads dataset.-------
excel = excel.rename(columns={
    "Aet": "Aet_excel",
    "Let": "Let_excel",
    "Ajorata": "AJORATA"
})


#-----From the roads dataset, we select the relevant columns.-------
roads = roads[[
    "LINK_ID",
    "LINK_MMLID",
    "TIENUMERO",
    "TIEOSANRO",
    "AET",
    "LET",
    "AJOSUUNTA",
    "AJORATA",
    "geometry"
]]

#---------We rename the columns in the roads dataset to match the excel dataset for merging.---------
roads = roads.rename(columns={
    "TIENUMERO": "Tie",
    "TIEOSANRO": "Aosa"
})

print("road MODDED", roads.columns.tolist())
print("excel MODDED", excel.columns.tolist())

In [ ]:
#--------Merging the two datasets on the "Tie" and "Aosa" columns, which represent the road number and segment number.---------
merged = roads.merge(excel, on=["Tie", "Aosa", "AJORATA"], how="inner")

#------After merging, we filter the merged dataset to only include rows where the AET and LET values
# from the roads dataset are within the Aet_excel and Let_excel values from the excel dataset.-------
merged = merged[
    (merged["AET"] <= merged["Let_excel"]) &
    (merged["LET"] >= merged["Aet_excel"])
].copy()


def get_effective_ajorata(kaista, ajorata):
    # Prefer real AJORATA if it is 1 or 2.
    # Otherwise infer direction from Kaista:
    # 11,12,13 -> 1
    # 21,22,23 -> 2
    if pd.notna(ajorata) and int(ajorata) in (1, 2):
        return int(ajorata)

    if pd.isna(kaista):
        return 1

    kaista = int(kaista)
    return kaista // 10
#------------------------------

#Converts Kaista values like 11,12,13 or 21,22 into lane index 1,2,3...
#Falls back safely if Kaista is missing.
def get_lane_index(kaista):
    # Extract lane number from Kaista:
    # 11 -> 1
    # 12 -> 2
    # 21 -> 1
    # 22 -> 2
    if pd.isna(kaista):
        return 1

    kaista = int(kaista)
    lane_index = kaista % 10

    if lane_index < 1:
        return 1
    return lane_index
#----------------------------

merged["effective_ajorata"] = merged.apply(
    lambda row: get_effective_ajorata(row["Kaista"], row["AJORATA"]),
    axis=1
)

merged["lane_index"] = merged["Kaista"].apply(get_lane_index)
#DEBUGGING
# print(merged[["AJORATA", "Kaista", "lane_index"]].head(10))

from shapely.geometry import LineString, MultiLineString

LANE_WIDTH = 3.5
LANE_GAP = 0.6

def offset_linestring(geom, distance):
    if geom is None or geom.is_empty:
        return geom
    try:
        if isinstance(geom, LineString):
            return geom.parallel_offset(distance, side="left", join_style=2)
        elif isinstance(geom, MultiLineString):
            parts = []
            for part in geom.geoms:
                off = part.parallel_offset(distance, side="left", join_style=2)
                if not off.is_empty:
                    parts.append(off)
            if not parts:
                return geom
            if len(parts) == 1:
                return parts[0]
            return MultiLineString(parts)
        return geom
    except Exception:
        return geom

def lane_offset_distance(effective_ajorata, lane_index):
    base = (lane_index - 1) * (LANE_WIDTH + LANE_GAP)
    return base if int(effective_ajorata) == 1 else -base

merged["lane_index"] = merged["Kaista"].apply(get_lane_index)

merged["lane_offset_m"] = merged.apply(
    lambda row: lane_offset_distance(row["effective_ajorata"], row["lane_index"]),
    axis=1
)

merged["geometry"] = merged.apply(
    lambda row: offset_linestring(row.geometry, row["lane_offset_m"]),
    axis=1
)

#------Droping the Aet_excel and Let_excel columns as they are no longer needed after filtering.----------
merged = merged.drop(columns=["Aet_excel", "Let_excel"])

#DEBUGGING
print(merged[["AJORATA", "Kaista", "lane_index", "lane_offset_m"]].head(20))

#------This was a proof of concept and will be heavily modified in the future.---------
merged["kunto_luokka"] = pd.cut(
    merged["RMS_yhd"],
    bins=[0,0.3,0.5,0.7,1],
    labels=["Erittäin hyvä", "Hyvä", "Tyydyttävä", "Huono"]
)

#-------Printing the shape and head of the merged dataset to verify the results before plotting and saving to file.---------
print(merged.shape)
print(merged.head())

#------Plotting the merged dataset with the "RMS_yhd" column as the color and a legend to show the different road conditions.---------
merged.plot(column="RMS_yhd", legend=True)


#------Here we define a function to convert the RMS_yhd values to color hex codes for better visualization in QGIS.--------
def rms_to_color(value):
    if pd.isna(value):
        return "#808080"  # Gray for NaN values
    elif value <= 0.3:
        return "#00FF00"  # Green for "Erittäin hyvä"
    elif value <= 0.5:
        return "#FFFF00"  # Yellow for "Hyvä"
    elif value <= 0.7:
        return "#FFA500"  # Orange for "Tyydyttävä"
    else:
        return "#FF0000"  # Red for "Huono"

#Applying the rms_to_color function to the "RMS_yhd" column to create a new "color_hex" column,
# that contains the corresponding color hex codes for each road segment based on its condition.
merged["color_hex"] = merged["RMS_yhd"].apply(rms_to_color)



#Saving the merged dataset to a GeoPackage file named "road_condition.gpkg".
#This is the file that will be used in the QGIS project to visualize the road conditions.
merged.to_file(output_folder / "road_condition.gpkg", driver="GPKG")


#This script is a proof of concept and will be heavily modified in the future to include more features and better visualization options.

#future ideas (feel free to add more):
# - Add filter system for the user to select the region of interest
# - Add dynamic file loading system to allow the user to select their own files for processing
# - Optimize performance.......
# - .......